# Predict an Iris flower
Load the existing trusted model, enter four measurements in centimeters, and display the species. No training is needed. Select the backend virtual environment as the notebook kernel.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

# Works from the project root, backend, or notebooks directory.
backend = next(
    folder
    for parent in [Path.cwd(), *Path.cwd().parents]
    for folder in [parent, parent / "backend"]
    if (folder / "models" / "iris_decision_tree.joblib").is_file()
)
metadata = json.loads((backend / "models/model_metadata.json").read_text())
model = joblib.load(backend / "models/iris_decision_tree.joblib")
features = metadata["feature_names"]
labels = metadata["class_labels"]
assert list(model.feature_names_in_) == features
assert list(model.classes_) == list(range(len(labels)))
print("Input features:", features)


Input features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


## Enter measurements
Change these values to classify a different flower. Column order comes from the saved metadata. The saved pipeline applies its fitted preprocessing automatically.

In [2]:
sample = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}
inputs = pd.DataFrame([sample], columns=features)
predicted_class = int(model.predict(inputs)[0])
predicted_species = labels[predicted_class]
print("Predicted species:", predicted_species)


Predicted species: Iris Setosa


## Check the same input through FastAPI
TestClient calls the real application in this notebook without opening a port. The result must match the direct model prediction. For browser documentation, run Uvicorn and open http://localhost:8000/docs.

In [3]:
import sys
from fastapi.testclient import TestClient

if str(backend) not in sys.path:
    sys.path.insert(0, str(backend))
from app.main import create_app

with TestClient(create_app(backend)) as client:
    response = client.post("/api/predict", json=sample)
    assert response.status_code == 200, response.text
    assert response.json()["predicted_species"] == predicted_species
    print("API prediction:", response.json()["predicted_species"])


D:\MLDemo\backend\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
2026-11-09 16:53:30,503 INFO event=model_loaded version=1.0.0


2026-11-09 16:53:30,523 INFO HTTP Request: POST http://testserver/api/predict "HTTP/1.1 200 OK"


API prediction: Iris Setosa
